# KAMAR — Evidence-Aware AI Audit Agent

**Hackathon Prototype · Version 1.0**

KAMAR transforms an agent's response into an explainable audit decision: every claim must reference evidence, declared contradictions and execution traces are checked, and files can be verified via SHA-256 digests. The result is either `ALLOW` or `REVIEW`, along with associated reasons.

> This prototype checks structural consistency and presence of elements. It does not certify that a piece of evidence is true, nor does it detect undeclared contradictions.

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

PROJECT_NAME = "KAMAR-META-AGENT"
VERSION = "1.0.0"
OUTPUT_DIR = Path("kamar-meta-agent-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

payload = {
    "claims": [
        {"id": "C1", "text": "The document indicates a 12% increase.", "evidence_ids": ["E1"]},
        {"id": "C2", "text": "The result is guaranteed in all cases.", "evidence_ids": []},
    ],
    "evidence": [
        {"id": "E1", "description": "Excerpt from user-provided report."}
    ],
    "contradictions": [],
    "trace": [
        {"action": "request_received", "timestamp": "2026-11-03T09:00:00Z"},
        {"action": "audit_completed", "timestamp": "2026-11-03T09:00:02Z"},
    ],
}

print(f"{PROJECT_NAME} v{VERSION} · sample loaded · {len(payload['claims'])} claims")

KAMAR-META-AGENT v1.0.0 · sample loaded · 2 claims


## 2. Inspection Engine

Malformatted inputs trigger `REVIEW` findings instead of implicit approval. Duplicate evidence identifiers are flagged, missing references are listed, and the execution trace requires an action and a valid ISO timestamp.

In [9]:
class KamarInspector:
    def inspect(self, payload):
        findings = []
        if not isinstance(payload, dict):
            return {"decision": "REVIEW", "reasons": ["payload_must_be_object"], "claims": [], "contradictions": [], "trace": {"valid": False, "event_count": 0}, "artifacts": []}

        reasons = []
        evidence = payload.get("evidence", [])
        claims = payload.get("claims", [])
        contradictions = payload.get("contradictions", [])
        trace = payload.get("trace", [])
        if not isinstance(evidence, list): evidence = []; reasons.append("evidence_must_be_list")
        if not isinstance(claims, list): claims = []; reasons.append("claims_must_be_list")
        if not isinstance(contradictions, list): contradictions = []; reasons.append("contradictions_must_be_list")
        if not isinstance(trace, list): trace = []; reasons.append("trace_must_be_list")

        ids = [item.get("id") for item in evidence if isinstance(item, dict) and isinstance(item.get("id"), str) and item["id"].strip()]
        duplicates = sorted({item for item in ids if ids.count(item) > 1})
        if duplicates: reasons.append("duplicate_evidence_ids")
        evidence_ids = set(ids)
        for index, claim in enumerate(claims):
            if not isinstance(claim, dict):
                findings.append({"claim_index": index, "supported": False, "missing_evidence": [], "issue": "claim_must_be_object"})
                reasons.append("invalid_claim")
                continue
            refs = claim.get("evidence_ids", [])
            valid_refs = isinstance(refs, list) and all(isinstance(ref, str) and ref.strip() for ref in refs)
            missing = [ref for ref in refs if ref not in evidence_ids] if valid_refs else []
            supported = valid_refs and bool(refs) and not missing and bool(str(claim.get("text", "")).strip())
            issue = None if supported else ("invalid_evidence_references" if not valid_refs else "missing_or_empty_evidence")
            findings.append({"claim_id": claim.get("id", f"claim_{index+1}"), "text": claim.get("text", ""), "supported": bool(supported), "missing_evidence": missing, "issue": issue})
            if not supported: reasons.append("unsupported_claim")

        if contradictions: reasons.append("reported_contradictions")
        trace_valid = bool(trace) and all(isinstance(event, dict) and isinstance(event.get("action"), str) and event["action"].strip() and self._valid_timestamp(event.get("timestamp")) for event in trace)
        if not trace_valid: reasons.append("invalid_or_empty_trace")
        artifact_results = []
        for artifact in payload.get("artifacts", []):
            result = self.verify_artifact(artifact)
            artifact_results.append(result)
            if not result["integrity"]: reasons.append("artifact_integrity_failure")

        reasons = list(dict.fromkeys(reasons))
        return {"decision": "REVIEW" if reasons else "ALLOW", "reasons": reasons, "claims": findings, "contradictions": contradictions, "trace": {"valid": trace_valid, "event_count": len(trace)}, "duplicate_evidence_ids": duplicates, "artifacts": artifact_results}

    @staticmethod
    def _valid_timestamp(value):
        if not isinstance(value, str): return False
        try:
            datetime.fromisoformat(value.replace("Z", "+00:00"))
            return True
        except ValueError:
            return False

    @staticmethod
    def verify_artifact(artifact):
        if not isinstance(artifact, dict) or not artifact.get("path"):
            return {"path": None, "exists": False, "integrity": False, "error": "invalid_artifact_spec"}
        path = Path(artifact["path"])
        if not path.is_file():
            return {"path": str(path), "exists": False, "integrity": False, "error": "file_not_found"}
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        expected = artifact.get("sha256")
        ok = isinstance(expected, str) and digest.lower() == expected.lower()
        return {"path": str(path), "exists": True, "sha256": digest, "integrity": ok, "error": None if ok else "hash_missing_or_mismatch"}

inspector = KamarInspector()
result = inspector.inspect(payload)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "decision": "REVIEW",
  "reasons": [
    "unsupported_claim"
  ],
  "claims": [
    {
      "claim_id": "C1",
      "text": "The document indicates a 12% increase.",
      "supported": true,
      "missing_evidence": [],
      "issue": null
    },
    {
      "claim_id": "C2",
      "text": "The result is guaranteed in all cases.",
      "supported": false,
      "missing_evidence": [],
      "issue": "missing_or_empty_evidence"
    }
  ],
  "contradictions": [],
  "trace": {
    "valid": true,
    "event_count": 2
  },
  "duplicate_evidence_ids": [],
  "artifacts": []
}


## 3. Audit Report

The report includes the UTC timestamp, the prototype version, and the decision reasons. It is written to the local output directory.

In [10]:
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "framework": PROJECT_NAME,
    "version": VERSION,
    "scope_note": "Structural consistency of provided elements; no external semantic validation.",
    "result": result,
}
report_path = OUTPUT_DIR / "audit_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Report written: {report_path.resolve()}")

Report written: C:\EDPZ_Work\KAMAR_validation_resultats\kamar-meta-agent-output\audit_report.json


## 4. Quick Demonstration Checks

These assertions serve as consistency checks during the presentation: the incomplete case must return `REVIEW`, and a fully supported case must return `ALLOW`.

In [11]:
assert result["decision"] == "REVIEW"
assert "unsupported_claim" in result["reasons"]
assert result["trace"]["valid"] is True

valid_payload = {
    "claims": [{"id": "C1", "text": "Supported claim.", "evidence_ids": ["E1"]}],
    "evidence": [{"id": "E1", "description": "Demonstration piece."}],
    "contradictions": [],
    "trace": [{"action": "audit_completed", "timestamp": "2026-11-03T09:00:02Z"}],
}
valid_result = inspector.inspect(valid_payload)
assert valid_result["decision"] == "ALLOW", valid_result
print("Demonstration checks passed successfully.")
print("Incomplete case: REVIEW | Supported case: ALLOW")

Demonstration checks passed successfully.
Incomplete case: REVIEW | Supported case: ALLOW


## 5. Limitations and Next Steps

- `ALLOW` means the structural checks coded here have passed; it does not mean the claim is true.
- Contradictions are taken into account when declared in the input; their automated discovery will require a dedicated model or analysis rule.
- Before real-world usage, add a privacy policy, appropriate key management for deployment, and adversarial testing.
- Recommended demo: show the unsupported claim, then add `E1` to its references and rerun the inspection.

In [12]:
# ============================================================
# KAMAR META-AGENT — FINAL DEMONSTRATION AND VALIDATION
# ============================================================

import json
import hashlib
import tempfile
from pathlib import Path

print("=" * 65)
print("KAMAR META-AGENT — FINAL VALIDATION")
print("=" * 65)

# 1. REVIEW CASE: unsupported claim
review_result = inspector.inspect(payload)

assert review_result["decision"] == "REVIEW", \
    "FAIL: incomplete payload should require REVIEW"

assert "unsupported_claim" in review_result["reasons"], \
    "FAIL: unsupported claim was not detected"

print("\n[PASS] REVIEW scenario")
print("Decision:", review_result["decision"])
print("Reasons:", review_result["reasons"])


# 2. ALLOW CASE: fully referenced claim and valid trace
allow_payload = {
    "claims": [
        {
            "id": "C1",
            "text": "The report indicates a 12 percent increase.",
            "evidence_ids": ["E1"]
        }
    ],
    "evidence": [
        {
            "id": "E1",
            "description": "Reference excerpt from the supplied report."
        }
    ],
    "contradictions": [],
    "trace": [
        {
            "action": "request_received",
            "timestamp": "2026-11-03T09:00:00Z"
        },
        {
            "action": "audit_completed",
            "timestamp": "2026-11-03T09:00:02Z"
        }
    ]
}

allow_result = inspector.inspect(allow_payload)

assert allow_result["decision"] == "ALLOW", \
    f"FAIL: expected ALLOW, got {allow_result}"

assert allow_result["claims"][0]["supported"] is True, \
    "FAIL: valid evidence reference was not accepted"

assert allow_result["trace"]["valid"] is True, \
    "FAIL: valid trace was rejected"

print("\n[PASS] ALLOW scenario")
print("Decision:", allow_result["decision"])
print("Claim supported:", allow_result["claims"][0]["supported"])
print("Trace valid:", allow_result["trace"]["valid"])


# 3. ARTIFACT INTEGRITY TEST
with tempfile.TemporaryDirectory() as temp_dir:
    test_file = Path(temp_dir) / "kamar_integrity_test.txt"
    test_file.write_text(
        "KAMAR integrity validation",
        encoding="utf-8"
    )

    expected_hash = hashlib.sha256(
        test_file.read_bytes()
    ).hexdigest()

    valid_artifact = inspector.verify_artifact({
        "path": str(test_file),
        "sha256": expected_hash
    })

    invalid_artifact = inspector.verify_artifact({
        "path": str(test_file),
        "sha256": "0" * 64
    })

    assert valid_artifact["integrity"] is True, \
        "FAIL: valid artifact hash rejected"

    assert invalid_artifact["integrity"] is False, \
        "FAIL: incorrect artifact hash accepted"

print("\n[PASS] Artifact integrity")
print("Valid hash accepted:", valid_artifact["integrity"])
print("Incorrect hash rejected:", not invalid_artifact["integrity"])


# 4. FINAL VALIDATION SUMMARY
summary = {
    "framework": PROJECT_NAME,
    "version": VERSION,
    "validation": {
        "review_scenario": review_result["decision"] == "REVIEW",
        "allow_scenario": allow_result["decision"] == "ALLOW",
        "valid_artifact": valid_artifact["integrity"],
        "invalid_hash_rejected": not invalid_artifact["integrity"]
    },
    "all_checks_passed": True
}

summary["all_checks_passed"] = all(
    summary["validation"].values()
)

summary_path = OUTPUT_DIR / "final_demo_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=True),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("FINAL RESULT")
print("=" * 65)

for name, passed in summary["validation"].items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nAll checks passed:", summary["all_checks_passed"])
print("Summary:", summary_path.resolve())

assert summary["all_checks_passed"], \
    "KAMAR final validation failed"

print("\nKAMAR DEMONSTRATION COMPLETED SUCCESSFULLY")

KAMAR META-AGENT — FINAL VALIDATION

[PASS] REVIEW scenario
Decision: REVIEW
Reasons: ['unsupported_claim']

[PASS] ALLOW scenario
Decision: ALLOW
Claim supported: True
Trace valid: True

[PASS] Artifact integrity
Valid hash accepted: True
Incorrect hash rejected: True

FINAL RESULT
review_scenario: PASS
allow_scenario: PASS
valid_artifact: PASS
invalid_hash_rejected: PASS

All checks passed: True
Summary: C:\EDPZ_Work\KAMAR_validation_resultats\kamar-meta-agent-output\final_demo_summary.json

KAMAR DEMONSTRATION COMPLETED SUCCESSFULLY
